# Week 5 Assignment

### Apache Spark Fundamentals and Data Processing using DataFrames

**Intern Name:** Sanjana

**Technology:** Apache Spark (PySpark)

**Objective:** Data Cleaning, Transformation, Filtering, Aggregation and Data Processing using Spark DataFrames.

### Objective
The objective of this assignment is to understand the fundamentals of Apache Spark and perform data cleaning, transformation, filtering, aggregation, and applying transformations, and generating meaningful insights from the data.

## Assignment Organization

To maintain a logical learning flow and follow the internship guidelines, the questions from the assignment have been integrated within the corresponding implementation steps.

The notebook is organized as follows:

* Step 1: Understanding Spark Basics (Q1, Q2)
* Step 2: Starting Apache Spark
* Step 3: Loading and Exploring Dataset
* Step 4: Data Cleaning (Q3, Q5, Q9, Q12)
* Step 5: Data Filtering (Q8)
* Step 6: Data Transformation (Q7, Q10)
* Step 7: Aggregation Operations (Q13)
* Step 8: Grouping Data and Analysis (Q4, Q6, Q11)
* Step 9: Advanced Concepts (Q14)
* Step 10: Complete Data Processing Pipeline (Q15)

This structure follows the practical workflow of data processing while ensuring that all assignment questions and objectives are covered systematically.


## Step 1 : Understanding Spark Basics

Traditional MapReduce processed data by reading from and writing to disk after every stage of execution. This disk-based processing increases execution time, especially for iterative algorithms and interactive analytics.

Apache Spark overcomes these limitations by performing most computations in memory, significantly reducing disk I/O operations. Spark also provides high-level APIs, supports multiple programming languages, and offers built-in libraries for SQL, machine learning, graph processing, and streaming applications.

## Step 2 : Starting Apache Spark

### Importing Required Libraries

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Creating Spark Session
SparkSession is the entry point of every Spark application.It allows us to create DataFrames and perform distributed data processing.

In [2]:
spark = SparkSession.builder \
    .appName("Week5_Spark_Assignment") \
        .getOrCreate()
print("Spark Session Created Successfully")

Spark Session Created Successfully


## Step 3 : Loading and Exploring Dataset

### Loading the Dataset
The dataset is stored inside the **data** folder. The header option reads the first row as column names, while inferSchema automatically detects appropriate data types.

In [3]:
df = spark.read.csv(
    "../data/dataset.csv",
    header=True,
    inferSchema=True
)

## Displaying Dataset
Before performing any transformation, it is important to understand the structure and contents of the dataset.

The following operations display:
- First few records
- Column names 
- Data types 
- Total number of records

In [4]:
df.show(10)
df.printSchema()
print("Total Records :", df.count())

+-------+----------------+------+----------------+-----------+-----------+---+------------+--------+-------+--------+---------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|       city|age|subscription|  status|  price|store_id|          email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+-----------+---+------------+--------+-------+--------+---------------+--------+-------------------+
|      1|      2024-05-14|  West|       Furniture|    1829.84|     Mumbai| 54|       Basic|Inactive|1829.84|     104| user1@mail.com|   user1|2024-02-17 09:17:00|
|      2|      2024-08-24|  East|       Furniture|    1771.78|    Kolkata| 55|     Premium|  Active|1771.78|     104|           NULL|   user2|2024-01-07 07:46:00|
|      3|      2024-04-26| North|         Grocery|     343.26| Chandigarh| 59|     Premium|  Active| 343.26|     104|           NULL|   user3|2024-06-19 20:55:00|
|      4|      2024-02

## Observation

The dataset has been successfully loaded into a Spark DataFrame.

The schema confirms that Spark has automatically identified the data types of each column, and the initial records provide a clear understanding of the dataset structure before further processing.

## Step 4 : Data Cleaning

In [5]:
# Removing duplicate records
df = df.dropDuplicates(["user_id","transaction_date"])
print("Total Records after removing duplicates:") 
print(df.count())

# Handling Missing Values
df = df.na.fill(
    {
        "status":"Unknown",
        "price" : 0
    }
)

# Removing invalid records
df = df.filter(
    (col("email").isNotNull()) &
    (col("username") != "")
)

df.show(5)

Total Records after removing duplicates:
160
+-------+----------------+------+----------------+-----------+----------+---+------------+--------+-------+--------+--------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|      city|age|subscription|  status|  price|store_id|         email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+----------+---+------------+--------+-------+--------+--------------+--------+-------------------+
|      1|      2024-05-14|  West|       Furniture|    1829.84|    Mumbai| 54|       Basic|Inactive|1829.84|     104|user1@mail.com|   user1|2024-02-17 09:17:00|
|      5|      2024-07-02| South|        Clothing|     1887.6| Bengaluru| 45|       Basic| Unknown|    0.0|     104|user5@mail.com|   user5|2024-10-15 08:11:00|
|      6|      2024-11-09|  West|     Electronics|     383.35|    Mumbai| 35|     Premium|  Active| 383.35|     103|user6@mail.com|   user6|2024-08-20

### Observation

Duplicate records were removed, missing values were handled, and invalid records were filtered from the dataset. These cleaning operations improved data quality and prepared the dataset for further analysis.

## Step 5 : Filtering Data
Data filtering is used to extract records that satisfy specific conditions. It helps in analyzing a particular subset of data instead of processing the entire dataset.

In this step, different filtering operations are performed based on region, category, age, and subscription type.

### Filtering Records by Region
Filtering records by region helps analyze data belonging to a specific geographical area.

The following code displays all records where the region is West.

In [6]:
df.filter(col("region")=="West").show()

+-------+----------------+------+----------------+-----------+------+---+------------+--------+-------+--------+---------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|  city|age|subscription|  status|  price|store_id|          email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+------+---+------------+--------+-------+--------+---------------+--------+-------------------+
|      1|      2024-05-14|  West|       Furniture|    1829.84|Mumbai| 54|       Basic|Inactive|1829.84|     104| user1@mail.com|   user1|2024-02-17 09:17:00|
|      6|      2024-11-09|  West|     Electronics|     383.35|Mumbai| 35|     Premium|  Active| 383.35|     103| user6@mail.com|   user6|2024-08-20 04:54:00|
|     17|      2024-09-25|  West|     Electronics|    2536.85|  Pune| 30|     Premium|Inactive|2536.85|     103|user17@mail.com|  user17|2024-07-16 13:03:00|
|     20|      2024-06-28|  West|         Grocery|  

### Filtering Records by Product Category
Filtering by product category allows us to analyze records related to a specific category.

The following example displays only Electronics products.

In [7]:
df.filter(col("product_category") == "Electronics").show()

+-------+----------------+------+----------------+-----------+-----------+---+------------+--------+-------+--------+---------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|       city|age|subscription|  status|  price|store_id|          email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+-----------+---+------------+--------+-------+--------+---------------+--------+-------------------+
|      6|      2024-11-09|  West|     Electronics|     383.35|     Mumbai| 35|     Premium|  Active| 383.35|     103| user6@mail.com|   user6|2024-08-20 04:54:00|
|     10|      2024-10-12| South|     Electronics|     971.59|  Bengaluru| 46|       Basic|  Active| 971.59|     105|user10@mail.com|  user10|2024-06-21 13:54:00|
|     13|      2024-11-23|  East|     Electronics|    3520.54|Bhubaneswar| 21|       Basic| Unknown|3520.54|     105|user13@mail.com|  user13|2024-06-14 04:07:00|
|     14|      2024-10

In [8]:
# Filter by Premium users
df.filter(
    (col("age")>=18) &
    (col("age")<=30)&
    (col("subscription")=="Premium")
).show()

+-------+----------------+------+----------------+-----------+-----------+---+------------+--------+-------+--------+----------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|       city|age|subscription|  status|  price|store_id|           email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+-----------+---+------------+--------+-------+--------+----------------+--------+-------------------+
|     17|      2024-09-25|  West|     Electronics|    2536.85|       Pune| 30|     Premium|Inactive|2536.85|     103| user17@mail.com|  user17|2024-07-16 13:03:00|
|     30|      2024-05-13|  East|       Furniture|    1813.97|Bhubaneswar| 19|     Premium|Inactive|1813.97|     103| user30@mail.com|  user30|2024-06-18 17:29:00|
|     36|      2024-10-18|  West|         Grocery|    2251.18|       Pune| 19|     Premium|  Active|2251.18|     102| user36@mail.com|  user36|2024-06-16 05:16:00|
|     43|      2

## Step 6 : Data Transformation

Data transformation is the process of modifying data into a more suitable format for analysis.

Common transformation operations include changing data types, renaming columns, creating new columns, and modifying existing values.

In this step, a timestamp column is converted to the appropriate data type and renamed for better readability.

In [9]:
from pyspark.sql.functions import col, to_timestamp

# Convert raw_timestamp column into TimestampType
df = df.withColumn(
    "raw_timestamp",
    to_timestamp
    (
        col("raw_timestamp"),
        "dd-MM-yyyy HH:mm"
    )
)

# Rename column
df = df.withColumnRenamed(
    "raw_timestamp",
    "event_time"
)

# Display updated schema
df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = false)
 |-- price: double (nullable = false)
 |-- store_id: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [10]:
df = df.withColumn(
    "age",
    col("age").cast(IntegerType())
)
df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = false)
 |-- price: double (nullable = false)
 |-- store_id: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



### Observation

The raw_timestamp column was converted into TimestampType and renamed to event_time. The age column was also converted to IntegerType, improving schema consistency and data usability.

## Step 7 : Aggregation

Aggregation is used to summarize large datasets into meaningful statistical information.

Instead of analyzing every record individually, aggregation functions help calculate totals, averages, minimum values, maximum values, and record counts.

These operations provide quick insights into the overall characteristics of the dataset.

### Basic Aggregation Operations

The following aggregation functions are used:

- count() → Counts total records
- sum() → Calculates total value
- avg() → Calculates average value
- min() → Finds minimum value
- max() → Finds maximum value

These functions help summarize the dataset efficiently.

In [11]:
from pyspark.sql.functions import count, sum, avg, min, max

df.select(
    count("*").alias("Total Records"),
    sum("sale_amount").alias("Total Sales"),
    avg("sale_amount").alias("Average Sales"),
    min("sale_amount").alias("Minimum Sales"),
    max("sale_amount").alias("Maximum Sales"),
).show()

+-------------+------------------+------------------+-------------+-------------+
|Total Records|       Total Sales|     Average Sales|Minimum Sales|Maximum Sales|
+-------------+------------------+------------------+-------------+-------------+
|          143|363763.56000000006|2543.8011188811192|        126.7|      4972.54|
+-------------+------------------+------------------+-------------+-------------+



In [12]:
# Aggregation implementation
df.agg(
    min("price").alias("Minimum Price"),
    max("price").alias("Maximum price"),
    avg("price").alias("Average Price")
).show()

+-------------+-------------+------------------+
|Minimum Price|Maximum price|     Average Price|
+-------------+-------------+------------------+
|          0.0|      4972.54|2071.5854545454554|
+-------------+-------------+------------------+



In [13]:
df.select(
    sum("sale_amount").alias("Total Revenue")
).show()

+------------------+
|     Total Revenue|
+------------------+
|363763.56000000006|
+------------------+



### Observation

Aggregation functions provided important statistical insights such as total sales, average sales, minimum values, maximum values, and total revenue from the dataset.

## Step 8 : Grouping Data

Grouping is one of the most important operations in Spark DataFrames.

The groupBy() function is used to group records that have similar values in a particular column. Once the data is grouped, aggregation functions such as count(), sum(), and avg() can be applied to generate meaningful insights.

Grouping helps transform raw data into summarized information that is easier to analyze and interpret.

### Grouping Data by Product Category

The following operation groups records by product category and calculates the average sale amount for each category.

In [14]:
df.groupBy(
    "product_category"
).avg(
    "sale_amount"
).show()

+----------------+------------------+
|product_category|  avg(sale_amount)|
+----------------+------------------+
|         Grocery| 2202.100000000001|
|     Electronics|2486.1003448275865|
|        Clothing| 2859.371136363636|
|       Furniture|2546.4329411764706|
+----------------+------------------+



### Grouping Data by Store

Grouping can also be used to calculate total sales for each store.

The following query groups records by store_id and calculates the total revenue generated by each store.

In [15]:
df.groupBy(
    "store_id"
).sum(
    "sale_amount"
).show()

+--------+-----------------+
|store_id| sum(sale_amount)|
+--------+-----------------+
|     101|          94270.8|
|     103|68274.56999999998|
|     102|68782.20000000001|
|     105|         55980.91|
|     104|76455.08000000002|
+--------+-----------------+



In [16]:
#West region category analyzes
df.filter(
    col("region") == "West"
).groupBy(
    "product_category"
).avg(
    "sale_amount"
).show()

+----------------+------------------+
|product_category|  avg(sale_amount)|
+----------------+------------------+
|         Grocery|2234.9605882352944|
|     Electronics|2886.4271428571433|
|        Clothing|2691.7277777777776|
|       Furniture| 2452.813333333333|
+----------------+------------------+



In [17]:
# City count
df.groupBy(
    "city"
).count().filter(
    col("count")>100
).show()

+----+-----+
|city|count|
+----+-----+
+----+-----+



### Observation

Grouping operations summarized the dataset by product category, store, and city. These results helped identify trends and compare performance across different groups.

## Step 9 : Advanced Concepts

Apache Spark automatically determines data types when loading a dataset using the inferSchema=True option.

Although this feature is very useful, it can sometimes create issues when the source data contains inconsistent or messy values.

Understanding such limitations is important for building reliable data processing pipelines.


The `inferSchema=True` option allows Spark to automatically detect the data type of each column while loading a dataset.

However, when a dataset contains inconsistent date formats, Spark may incorrectly infer the column type or interpret values incorrectly.

For example:

```text
12-05-2024
2024/05/12
May 12, 2024
```

All three values represent dates but use different formats.

In such cases, Spark may:

* Interpret some values as strings
* Fail to recognize valid dates
* Produce null values after conversion
* Generate inconsistent schemas

Therefore, it is considered a good practice to clean and standardize date formats before performing schema inference or explicitly define the schema when loading important datasets.


### Observation

Understanding schema inference, shuffle operations, and wide transformations helps in designing efficient Spark applications and avoiding common data processing issues.

## Step 10 : Complete Data Processing Pipeline

In real-world projects, data processing is usually performed through a pipeline where multiple operations are combined together.

Instead of executing each step separately, Spark allows us to chain operations together to create an efficient and readable workflow.

In this section, a complete pipeline is created that:

Removes duplicate records
Handles missing values
Groups records by store_id
Calculates total revenue for each store

This demonstrates a practical end-to-end Spark DataFrame workflow.

In [18]:
pipeline_df = (
    df
    .dropDuplicates()
    .na.fill({"price": 0})
    .groupBy("store_id")
    .agg(
        sum("sale_amount").alias("Total Revenue")
    )
)

pipeline_df.show()

+--------+-----------------+
|store_id|    Total Revenue|
+--------+-----------------+
|     101|          94270.8|
|     103|68274.56999999998|
|     102|68782.20000000001|
|     105|         55980.91|
|     104|76455.08000000002|
+--------+-----------------+



In [19]:
import pandas as pd

# Convert Spark DataFrame to Pandas DataFrame
results_df = pipeline_df.toPandas()

# Save output file
results_df.to_csv("../output/results.csv", index=False)

print("results.csv saved successfully")

results.csv saved successfully


### Observation

The complete pipeline successfully combined cleaning, transformation, grouping, and aggregation operations into a single workflow, demonstrating a practical Spark data processing solution.

# Final Conclusion

In this assignment, Apache Spark DataFrame operations were successfully implemented to perform data cleaning, filtering, transformation, aggregation, grouping, and pipeline creation.

The dataset was explored, duplicate records were removed, missing values were handled, and inconsistent data was identified and cleaned. Various filtering and transformation techniques were applied to prepare the data for analysis.

Aggregation and grouping operations were then used to generate meaningful insights from the dataset. Finally, a complete data processing pipeline was developed by combining multiple Spark operations into a single workflow.

Overall, this assignment provided practical experience with Apache Spark DataFrames and demonstrated how Spark can efficiently process and analyze large datasets for real-world applications.
